In [1]:
import sys
sys.path.append("..")

import mlflow
from mlflow import MlflowClient
from src.datos import cargar_obt, agregar_features
from src.pipeline import preparar_datos, configurar_mlflow, FEATURES, TARGET, wape

configurar_mlflow()
client = MlflowClient()

modelo = mlflow.sklearn.load_model("models:/rossmann-modelo@champion")
version = client.get_model_version_by_alias("rossmann-modelo", "champion")
registrado = client.get_run(version.run_id).data.metrics["wape_prueba"]

_, prueba = preparar_datos(agregar_features(cargar_obt()))
reproducido = wape(prueba[TARGET], modelo.predict(prueba[FEATURES]))

print("Versión:", version.version, "| run_id:", version.run_id)
print("WAPE registrado: ", registrado)
print("WAPE reproducido:", reproducido)
print("¿Idénticos?", reproducido == registrado)

Versión: 1 | run_id: 0e34d952530740ec900db92ec5831a15
WAPE registrado:  0.09023300468244734
WAPE reproducido: 0.09023300468244734
¿Idénticos? True


In [2]:
import pandas as pd

todas = mlflow.search_runs(experiment_names=["rossmann"], run_view_type=mlflow.entities.ViewType.ACTIVE_ONLY)
todas["inicio"] = pd.to_datetime(todas["start_time"]).dt.tz_convert("America/Bogota")

reejecucion = pd.Timestamp("2026-11-18 10:00", tz="America/Bogota")
sobran = todas[
    (todas["status"] != "FINISHED")
    | (todas["tags.mlflow.runName"].str.contains("prueba_hgb", na=False))
    | (todas["tags.fase"].isin(["baseline", "candidato"]) & (todas["inicio"] < reejecucion))
]
print("Corridas activas:", len(todas), "| a borrar:", len(sobran))
sobran[["tags.mlflow.runName", "status", "inicio"]].sort_values("inicio")

Corridas activas: 107 | a borrar: 0


,tags.mlflow.runName,status,inicio


In [3]:
for run_id in sobran["run_id"]:
    client.delete_run(run_id)

activas = mlflow.search_runs(experiment_names=["rossmann"])
print("Corridas activas después de limpiar:", len(activas))

Corridas activas después de limpiar: 107


In [4]:
from pathlib import Path

corridas = mlflow.search_runs(experiment_names=["rossmann"])
Path("../docs").mkdir(exist_ok=True)
corridas.to_csv("../docs/corridas.csv", index=False)

print("Exportadas:", len(corridas), "corridas con", corridas.shape[1], "columnas")
print(corridas["tags.fase"].value_counts())

Exportadas: 107 corridas con 149 columnas
tags.fase
tuning       96
candidato     8
baseline      2
final         1
Name: count, dtype: int64
